In [13]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

from ipywidgets import widgets,interact,HBox


In [14]:
me = 9.11e-31
h = 6.626e-34
kB = 1.38e-23
JpereV = 1.6e-19

In [15]:
def gCB(E,EC,mass):
    return JpereV*np.sqrt(128)*np.pi*(mass*me)**1.5/h**3*np.sqrt(np.heaviside(E-EC,0)*(E-EC)*JpereV)

def gVB(E,EV,mass):
    return JpereV*np.sqrt(128)*np.pi*(mass*me)**1.5/h**3*np.sqrt(np.heaviside(EV-E,0)*(EV-E)*JpereV)

def f(E,EF,T):
    return 1./(1.+np.exp((E-EF)*JpereV/kB/T))

In [16]:
E = np.linspace(0,10,num=1600)

In [17]:
all = go.FigureWidget(
    data=(
        go.Scatter(
            name='E_F',
            marker=dict(color='blue'),
            x=[1,1e30],
            y=[3.5]*2,
        ),
        go.Scatter(
            name='gCB',
            marker=dict(color='DarkGreen'),
            x=gCB(E,5.0,1.0),
            y=E,
        ),
        go.Scatter(
            name='gVB',
            marker=dict(color='DarkRed'),
            x=gVB(E,3.0,1.0),
            y=E,
        ),
        go.Scatter(
            name='nCB',
            marker=dict(color='LightGreen'),
            x=f(E,4.0,1000)*gCB(E,5.0,1.0),
            y=E,
        ),
        go.Scatter(
            name='pVB',
            marker=dict(color='salmon'),
            x=(1-f(E,4.0,1000))*gVB(E,3.0,1.0),
            y=E,
        ),
    ),
    layout=dict(template='seaborn',
                title='overview',
                height=600,
                width=300,
                xaxis=dict(range=[20,30],
                           type='log',
                          ),
                yaxis_range=[np.min(E),np.max(E)],
               )
)

fermi = go.FigureWidget(
    data=(
        go.Scatter(
            name='E_F',
            marker=dict(color='blue'),
            x=[0,1],
            y=[3.5]*2,
        ),
        go.Scatter(
            name='f',
            marker=dict(color='green'),
            x=f(E,4.0,1000),
            y=E,
        ),
        go.Scatter(
            name='1-f',
            marker=dict(color='red'),
            x=1-f(E,4.0,1000),
            y=E,
        ),
    ),
    layout=dict(template='seaborn',
                title='Fermi–Dirac',
                height=600,
                width=300,
                xaxis=dict(range=[0,1],
                           type='linear',
                          ),
                yaxis_range=[np.min(E),np.max(E)],
               )
)

density = go.FigureWidget(
    data=(
        go.Scatter(
            name='E_F',
            marker=dict(color='blue'),
            x=[0,1e28],
            y=[3.5]*2,
        ),
        go.Scatter(
            name='gCB',
            marker=dict(color='DarkGreen'),
            x=gCB(E,5.0,1.0),
            y=E,
        ),
        go.Scatter(
            name='gVB',
            marker=dict(color='DarkRed'),
            x=gVB(E,3.0,1.0),
            y=E,
        ),
    ),
    layout=dict(template='seaborn',
                title='density',
                height=600,
                width=300,
                xaxis=dict(range=[0,1e28],
                           type='linear',
                          ),
                yaxis_range=[np.min(E),np.max(E)],
               )
)

occupancy = go.FigureWidget(
    data=(
        go.Scatter(
            name='E_F',
            marker=dict(color='blue'),
            x=[0,1e26],
            y=[3.5]*2,
        ),
        go.Scatter(
            name='nCB',
            marker=dict(color='LightGreen'),
            x=f(E,4.0,1000)*gCB(E,5.0,1.0),
            y=E,
        ),
        go.Scatter(
            name='pVB',
            marker=dict(color='salmon'),
            x=(1-f(E,4.0,1000))*gVB(E,3.0,1.0),
            y=E,
        ),
    ),
    layout=dict(template='seaborn',
                title='occupancy',
                height=600,
                width=300,
                xaxis=dict(range=[0,1e26],
                           type='linear',
                          ),
                yaxis_range=[np.min(E),np.max(E)],
               )
)


In [23]:
@interact
def view_data(T=widgets.FloatSlider(value=1000,min=0.0,max=3000,step=50,description='temperature'),
              masse=widgets.FloatSlider(value=1.0,min=0.1,max=2.0,step=0.1,description='m_eff electron'),
              massh=widgets.FloatSlider(value=1.0,min=0.1,max=2.0,step=0.1,description='m_eff hole'),
              EC=widgets.FloatSlider(value=5.0,min=0.0,max=10.0,step=0.1,description='E_C'),
              EV=widgets.FloatSlider(value=3.0,min=0.0,max=10.0,step=0.1,description='E_V'),
              dE=widgets.FloatSlider(value=0.0,min=-1.0,max=1.0,step=0.1,description='ΔE_F'),
              noMassEF=widgets.Checkbox(value=False,description='E_F independent of m_e/m_h'),
             ):
    with all.batch_update():
        EF = dE+np.average([EC,EV])-(0.0 if noMassEF else 0.75*kB*T*np.log(masse/massh)/JpereV)
        all.data[0].y = [EF]*2
        all.data[1].x = gCB(E,EC,masse)
        all.data[2].x = gVB(E,EV,massh)
        all.data[3].x = f(E,EF,T)*gCB(E,EC,masse)
        all.data[4].x = (1-f(E,EF,T))*gVB(E,EV,massh)
        fermi.data[0].y = [EF]*2
        fermi.data[1].x = f(E,EF,T)
        fermi.data[2].x = 1-f(E,EF,T)
        density.data[0].y = [EF]*2
        density.data[1].x = gCB(E,EC,masse)
        density.data[2].x = gVB(E,EV,massh)
        occupancy.data[0].y = [EF]*2
        occupancy.data[1].x = f(E,EF,T)*gCB(E,EC,masse)
        occupancy.data[2].x = (1-f(E,EF,T))*gVB(E,EV,massh)

HBox([all,fermi,density,occupancy])

interactive(children=(FloatSlider(value=1000.0, description='temperature', max=3000.0, step=50.0), FloatSlider…

    'data': [{'marker': {'color': 'blue'},
              'name': 'E_F',
        …